In [1]:
platformID = 'INS'

In [2]:
from datetime import datetime, date
import pandas as pd
import numpy as np
import os 

import psycopg2
import missingno as msno

## import helper 

In [1]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader, execute_sql_query, compare_or_update_reference, fix_country_custom_pct_rel
import test_functions 

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes', 'CountryName',])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_00 passed: lookup DataFrame is not empty.
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_01 passed: No duplicate ['w/c'].
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_02 passed: No missing values in lookup.
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_03 passed: lookup DataFrame is not empty.
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_04 passed: No duplicate ['Channel ID'].
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_05 passed: No missing values in lookup.
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_06 passed: lookup DataFrame is not empty.
[2026-03-31 12:37:26] [INFO] ...updating logbook...

[2026-03-31 12:37:26] [INFO] ✅ Test INS_1c_07 pas

## functions 

In [5]:
def raw_processing(ig_raw: pd.DataFrame,
                     country_codes: pd.DataFrame) -> pd.DataFrame:
    """
    Clean Instagram raw country data and fill missing YT-_FBE_codes values.

    This function:
    1. Loads and standardises base columns (Channel ID, Channel Name, w/c).
    2. Ensures YT-_FBE_codes is a string with no NaN values.
    3. Performs prioritised deduplication:
       - If duplicate rows exist (same w/c, Channel ID, country, demographics)
         keep the row where YT-_FBE_codes is *not* empty.
    4. Splits the data into:
         A) rows with existing codes
         B) rows missing codes
    5. Normalises ambiguous or non-standard country names.
    6. Merges missing rows back with the country_codes reference table.
    7. Recombines all rows and returns a DataFrame
       guaranteed to have *no missing YT-_FBE_codes*.

    Parameters
    ----------
    ig_raw : pd.DataFrame
        Raw Instagram country-level extract containing columns:
        - page_id
        - page_name
        - week_commencing
        - country_code
        - ins_country_name
        - followers_by_demographic

    country_codes : pd.DataFrame
        A lookup table with columns:
        - CountryName
        - YT-_FBE_codes

    Returns
    -------
    pd.DataFrame
        Fully cleaned DataFrame with:
        - Standardised column names
        - Deduplicated rows (keeping coded versions)
        - Normalised country names
        - Filled missing YT-_FBE_codes
        - Guaranteed no missing or empty country codes
    """

    ig = ig_raw.copy()

    # --- Standardise base columns ---
    ig['page_id'] = ig['page_id'].astype(str)
    ig['week_commencing'] = pd.to_datetime(ig['week_commencing'])

    ig = ig.rename(columns={
        'page_id': 'Channel ID',
        'page_name': 'Channel Name',
        'week_commencing': 'w/c',
        'country_code': 'YT-_FBE_codes',
    })

    ig['YT-_FBE_codes'] = ig['YT-_FBE_codes'].fillna('')

    # --- Prioritised deduplication ---
    ig = (
        ig.assign(code_empty=lambda df: df['YT-_FBE_codes'].eq(''))
          .sort_values('code_empty')  # valid codes first
          .drop_duplicates(
              subset=['w/c', 'Channel ID', 'ins_country_name', 'followers_by_demographic'],
              keep='first'
          )
          .drop(columns='code_empty')
    )

    # --- Split found vs missing codes ---
    found = ig.loc[ig['YT-_FBE_codes'] != ''].copy()
    still_missing = ig.loc[ig['YT-_FBE_codes'] == ''].copy()

    # --- Normalise country names ---
    mapping = {
        'Brunei Darussalam': 'Brunei',
        'Congo (Democratic Republic of the)': 'Congo, Democratic Republic of the',
        'Congo (Republic of the)': 'Congo, Republic of',
        "Cote d'Ivoire": 'Ivory Coast',
        'Gambia (The)': 'Gambia',
        'Hong Kong (HKSAR)': 'Hong Kong',
        'Korea (South)': 'South Korea',
        'Kyrgyz Republic': 'Kyrgystan',
        'Lao PDR': 'Laos',
        'Macau (MSAR)': 'Macau',
        'Réunion': 'Reunion',
        'Russian Federation': 'Russia',
        'Slovak Republic': 'Slovakia',
        'Syrian Arab Republic': 'Syria',
        'Taiwan (Province of China)': 'Taiwan',
        'West Bank and Gaza': 'Palestinian Territories',
    }

    still_missing.loc[:, 'ins_country_name'] = \
        still_missing['ins_country_name'].replace(mapping)

    # --- Merge missing rows back with country_codes ---
    still_missing = (
        still_missing
        .drop(columns='YT-_FBE_codes')
        .merge(
            country_codes[['CountryName', 'YT-_FBE_codes']],
            left_on='ins_country_name',
            right_on='CountryName',
            how='left'
        )
        .drop(columns=['CountryName'])
    )

    # --- Recombine all rows ---
    out = pd.concat([found, still_missing], ignore_index=True)

    #--- Final validation: ensure no empty or NA codes remain ---
    if (out['YT-_FBE_codes'].isna() | out['YT-_FBE_codes'].eq('')).any():
        unresolved = out.loc[
            out['YT-_FBE_codes'].isna() | out['YT-_FBE_codes'].eq(''),
            'ins_country_name'
        ].unique()
        raise ValueError(f"Unresolved country codes for: {unresolved}")

    return out



## ingestion

In [6]:
# keep country code where clause - the other rows are for age & gender in diff cols
sql_query = f"""
    SELECT 
        week_commencing,
        l.ig_account_id as page_id,
        page_name, 
        country_code,
        country_name as ins_country_name,
        followers_by_demographic
    FROM
        redshiftdb.central_insights.adverity_social_instagram_by_page_demo AS p
    RIGHT JOIN
            world_service_audiences_insights.social_media_lookup_ig AS l
        ON 
            p.page_id = l.ig_identifier
    WHERE
            week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        AND
            followers_by_demographic > 0
        AND 
            country_name <> ''
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

ig_userCountry_raw = pd.read_csv(file, keep_default_na=False)

if not ig_userCountry_raw['page_id'].str.startswith(platformID, na=False).all():
    ig_userCountry_raw['page_id'] = platformID + ig_userCountry_raw['page_id']

ig_userCountry_raw = raw_processing(ig_userCountry_raw, country_codes)

### RUN TESTS
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
# missing page_ids
test_functions.test_filter_elements_returned(ig_userCountry_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=ig_userCountry_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(ig_userCountry_raw, 
                           numeric_columns=['followers_by_demographic'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in followers_by_demographic column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(ig_userCountry_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')



no redshift connection using last time queried


[2026-03-31 12:37:27] [INFO] ...updating logbook...



...testing Channel ID...
✅ Test INS_1c_09 passed: everything found!


[2026-03-31 12:37:28] [INFO] 
Summary of missing weeks per channel:
[2026-03-31 12:37:28] [INFO] 
          Channel ID  missing_week_count
INS17841435042594492                  50
INS17841402856323858                  10
INS17841464755735820                   9
INS17841400273561016                   2
INS17841400240564037                   2
INS17841400681780254                   2
INS17841400778089100                   2
INS17841400884455643                   2
INS17841400948656588                   2
INS17841401010530114                   2
INS17841401322662359                   2
INS17841401325222869                   2
INS17841401350693298                   2
INS17841401471775462                   2
INS17841401518836664                   2
INS17841401562994997                   2
INS17841401580814335                   2
INS17841401606325704                   2
INS17841400485756476                   2
INS17841400559646664                   2
INS17841400630063846                   2


✅ Test INS_1c_11 passed: No NaN and all numeric values > 0.
✅ Test INS_1c_12 passed: No combinations occurs more than once.


In [8]:
test_functions.test_key_consistency(ig_userCountry_raw, socialmedia_accounts, 
                               ['Channel ID'], 
                               f"{platformID}_1c_13",
                               test_step='checking social media accounts in lookup, adding service',
                               focus='left')
ig_userCountry = ig_userCountry_raw.merge(socialmedia_accounts[['Channel ID', 'ServiceID']], 
                                                      on='Channel ID', how='left')

[2026-03-31 12:38:07] [INFO] ...updating logbook...



✅ Test INS_1c_13 passed: No key mismatches.


In [9]:
test_functions.test_key_consistency(ig_userCountry, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_14",
                               test_step='calculating country %', 
                                focus='left')
ig_userCountry = ig_userCountry.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])

[2026-03-31 12:38:10] [INFO] ...updating logbook...



✅ Test INS_1c_14 passed: No key mismatches.


In [10]:
# currently sql table has duplicates
ig_userCountry = ig_userCountry.drop_duplicates()
# test for duplicate entries 
test_functions.test_duplicates(ig_userCountry, ['Channel ID', 'w/c', 'PlaceID'], 
                               test_number=f"{platformID}_1c_15",
                               test_step='dropped duplicates past processing')


[2026-03-31 12:38:14] [INFO] ...updating logbook...



✅ Test INS_1c_15 passed: No combinations occurs more than once.


In [ ]:
ig_userCountry = ig_userCountry[ig_userCountry['PlaceID'] != 'UNK']

grouped_df = ig_userCountry.groupby(['Channel ID', 'Channel Name', 
                                     'w/c']).agg({'followers_by_demographic': 'sum'}).reset_index()

# Rename the aggregated column
grouped_df = grouped_df.rename(columns={'followers_by_demographic': 'Sum_followers_by_demographic'})

right_cols = ['Channel ID', 'w/c', 'Sum_followers_by_demographic']
ig_country_df = ig_userCountry.merge(grouped_df[right_cols],
                                         how='inner',
                                         on=['Channel ID', 'w/c'])

test_functions.test_inner_join(ig_userCountry, grouped_df[right_cols], 
                               ['Channel ID', 'w/c'],
                               f"{platformID}_1c_16",)

ig_country_df['country_%'] = ig_country_df.apply(
    lambda row: row['followers_by_demographic'] / row['Sum_followers_by_demographic']
    if row['Sum_followers_by_demographic'] > 0 else 0, axis=1
)

test_functions.test_percentage(ig_country_df, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_17",
                               test_step='summing country % per week & account')

In [ ]:
ig_country_df.shape

In [ ]:
#sanity checks - previous weeks are not affected
# 4
display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 5
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 6 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())

#sanity checks 
# 1
display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 2
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 3 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())

#sanity checks 
# 1
display(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 2
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 3 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())



In [ ]:
### DANGER! FIXING IRAN REACH Manually DUE TO CURRENT SHUTDOWN 
def fix_country_custom_pct_abs(df, country_code, week_to_pct):
    """
    week_to_pct: dict mapping "YYYY-MM-DD" → forced percentage (0–1)
    """
    print("Fixing country percentages with mapping:")
    print(week_to_pct)

    df = df.copy()

    affected_weeks = set(week_to_pct.keys())
    affected = df[df['w/c'].isin(affected_weeks)]
    print("Affected rows:", affected.shape)

    for (channel, wc), group in affected.groupby(['Channel ID', 'w/c']):
        
        wc_str = str(wc.date())  # or str(wc)

        if wc_str not in week_to_pct:
            print(f"week not found: {wc_str}")
            continue
        
        new_country_pct = week_to_pct[wc_str]
        old_country_pct = group.loc[group['PlaceID'] == country_code, 'country_%']
        if old_country_pct.empty:
            continue

        old_country_pct = float(old_country_pct.iloc[0])

        # skip if already correct
        if abs(old_country_pct - new_country_pct) < 1e-9:
            continue

        # scale all OTHER countries proportionally
        scale_factor = (1 - new_country_pct) / (1 - old_country_pct)

        group_mask        = (df['Channel ID'] == channel) & (df['w/c'] == wc)
        country_mask      = group_mask & (df['PlaceID'] == country_code)
        non_country_mask  = group_mask & (df['PlaceID'] != country_code)

        df.loc[non_country_mask, 'country_%'] *= scale_factor
        df.loc[country_mask, 'country_%'] = new_country_pct

    return df

    
# Iran = PlaceID "IRN"
IRAN_CODE = "IRN"

ig_country_df = fix_country_custom_pct_rel(
    ig_country_df,
    IRAN_CODE,
    {
    "2026-01-12": 0.01,
    "2026-01-19": 0.67,
    "2026-03-02": 0.01,
    "2026-03-09": 0.01
}
)


In [ ]:
#sanity checks - previous weeks are not affected
# 4
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 5
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 6 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-05') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())

#sanity checks 
# 1
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 2
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 3 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-12') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())

#sanity checks 
# 1
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['PlaceID'] == 'IRN')].sort_values('country_%', ascending=False).head())
# 2
#display(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID'))
# 3 
print(ig_country_df[(ig_country_df['w/c'] == '2026-01-19') & (ig_country_df['Channel ID'] == 'INS17841400230391592')].sort_values('PlaceID')['country_%'].sum())



In [ ]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
ig_country_df[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_geog.csv", 
                           index=None)
'''
compare_or_update_reference(ig_country_df[cols], 
                            f"../test/refactoring/{platformID}_country_expected.pkl", 
                            cols, update=False)
'''